# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [18]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [6]:
print(f'Data size: {df.shape}')
print(f'Data column: {df.columns.tolist()}')
print(f'Data types: {df.dtypes}')

Data size: (1338, 7)
Data column: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
Data types: age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object


## A.2. Missing values & Duplicate data

In [9]:
print(f'Missing value: {df.isnull().sum()}')
print(f'Duplicate data: {df.duplicated().sum()}')

Missing value: age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
Duplicate data: 1


## A.3. Invalid values

In [13]:
print(f'Invalid values: {df.isna()}')


Invalid values:         age    sex    bmi  children  smoker  region  charges
0     False  False  False     False   False   False    False
1     False  False  False     False   False   False    False
2     False  False  False     False   False   False    False
3     False  False  False     False   False   False    False
4     False  False  False     False   False   False    False
...     ...    ...    ...       ...     ...     ...      ...
1333  False  False  False     False   False   False    False
1334  False  False  False     False   False   False    False
1335  False  False  False     False   False   False    False
1336  False  False  False     False   False   False    False
1337  False  False  False     False   False   False    False

[1338 rows x 7 columns]


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [24]:
df['bmi_group'] = np.select(
   [ 
        df['bmi'] < 25,
        (df['bmi'] >25) & (df['bmi'] <30),
        df['bmi'] >=30
   ],
   ['Normal','OverWeight','Obese'],
   default='Unknown'
)
print(df.bmi_group)


0       OverWeight
1            Obese
2            Obese
3           Normal
4       OverWeight
           ...    
1333         Obese
1334         Obese
1335         Obese
1336    OverWeight
1337    OverWeight
Name: bmi_group, Length: 1338, dtype: str


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [25]:
print(f'mean: {df.mean(numeric_only=True)}')    #mean, median, mode
print(f'Median: {df.median(numeric_only=True)}')
print(f'Mode: {df.mode(numeric_only=True)}')



mean: age            39.207025
bmi            30.663397
children        1.094918
charges     13270.422265
dtype: float64
Median: age           39.000
bmi           30.400
children       1.000
charges     9382.033
dtype: float64
Mode:    age   bmi  children    charges
0   18  32.3         0  1639.5631


## Group 2 — Dispersion

In [26]:
print(f'Do lech chuan: {df.std(numeric_only=True)}\n')
print(f'Phuong sai: {df.var(numeric_only=True)} ')
#range = max - min
numeric_df = df.select_dtypes(include='number')
print(f'Khoang bien thien:\n{numeric_df.max() - numeric_df.min()}')
#IQR: tứ phân vị = Q3 - Q1 
Q1 = numeric_df.quantile(0.25)
Q3 = numeric_df.quantile(0.75)
print(f'Tu Phan Vi: {Q3 -Q1}')


Do lech chuan: age            14.049960
bmi             6.098187
children        1.205493
charges     12110.011237
dtype: float64

Phuong sai: age         1.974014e+02
bmi         3.718788e+01
children    1.453213e+00
charges     1.466524e+08
dtype: float64 
Khoang bien thien:
age            46.00000
bmi            37.17000
children        5.00000
charges     62648.55411
dtype: float64
Tu Phan Vi: age            24.000000
bmi             8.397500
children        2.000000
charges     11899.625365
dtype: float64


## Group 3 — Location and Shape

In [27]:
print(f':Do lech: {numeric_df.skew()}\n')
print(f'Do nhon: {numeric_df.kurt()}')

:Do lech: age         0.055673
bmi         0.284047
children    0.938380
charges     1.515880
dtype: float64

Do nhon: age        -1.245088
bmi        -0.050732
children    0.202454
charges     1.606299
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [41]:
df_smoker = df.groupby('smoker')['charges'].sum()
print(df_smoker)
chech_lech = df_smoker['yes']/df_smoker['no']
print(f'người hút thuốc trả hơn {chech_lech} so với người không hút thuốc')

region_charges = df.groupby(['region', 'smoker'])['charges'].mean().unstack()

region_charges['ratio'] = region_charges['yes'] / region_charges['no']

print(region_charges)

smoker
no     8.974061e+06
yes    8.781764e+06
Name: charges, dtype: float64
người hút thuốc trả hơn 0.9785718041106573 so với người không hút thuốc
smoker              no           yes     ratio
region                                        
northeast  9165.531672  29673.536473  3.237514
northwest  8556.463715  30192.003182  3.528561
southeast  8032.216309  34844.996824  4.338155
southwest  8019.284513  32269.063494  4.023933


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [ ]:
# TODO


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [ ]:
# TODO


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [ ]:
# TODO


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [ ]:
# TODO


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*